# Task 1

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
np.random.seed(42)

n_rows = 500000

user_ids = np.arange(1, n_rows + 1)
cities_list = ["Berlin", "Tokyo", "New York", "London", "Paris", 
               "San Francisco", "Sydney", "Mumbai", "Toronto", "Seoul"]
cities = np.random.choice(cities_list, size=n_rows)
scores = np.random.uniform(0, 100, size=n_rows)
active_status = np.random.choice([True, False], size=n_rows)
end_date = datetime.now()
start_date = end_date - timedelta(days=3*365)

total_seconds = int((end_date - start_date).total_seconds())
random_offsets = np.random.randint(0, total_seconds, size=n_rows)
signup_dates = [start_date + timedelta(seconds=int(offset)) for offset in random_offsets]

ages = np.random.randint(18, 81, size=n_rows)

sessions = np.random.randint(0, 501, size=n_rows)

revenue = np.random.uniform(0, 1000, size=n_rows)
df = pd.DataFrame({
    'user_id': user_ids,
    'city': cities,
    'score': scores,
    'active': active_status,
    'signup_date': signup_dates,
    'age': ages,
    'sessions': sessions,
    'revenue': revenue
})

print(df.head())
print(df.info())

   user_id    city      score  active                signup_date  age  \
0        1  Sydney  43.689490   False 2025-01-13 10:30:35.780118   38   
1        2  London  97.403462    True 2024-01-28 02:48:56.780118   37   
2        3  Mumbai  66.148238   False 2023-12-07 19:19:43.780118   73   
3        4   Paris  21.993544   False 2026-01-08 23:22:39.780118   62   
4        5  Sydney  91.719953    True 2024-01-29 13:09:52.780118   20   

   sessions     revenue  
0       261  635.068666  
1        77  895.979998  
2       115  451.742873  
3       235  449.489098  
4       399  100.266323  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   user_id      500000 non-null  int64         
 1   city         500000 non-null  object        
 2   score        500000 non-null  float64       
 3   active       500000 non-null  bool      

In [3]:
df.head()

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Sydney,43.689490,False,2025-01-13 10:30:35.780118,38,261,635.068666
1,2,London,97.403462,True,2024-01-28 02:48:56.780118,37,77,895.979998
2,3,Mumbai,66.148238,False,2023-12-07 19:19:43.780118,73,115,451.742873
3,4,Paris,21.993544,False,2026-01-08 23:22:39.780118,62,235,449.489098
4,5,Sydney,91.719953,True,2024-01-29 13:09:52.780118,20,399,100.266323


**Step 2: Save the dataset as a Parquet file. Then use pyarrow.parquet.ParquetFile to inspect:
The number of row groups
The number of rows and columns
The schema (column names and types)
For each column in the first row group: physical type, compressed size, and any available statistics (min, max, null count)**

In [14]:
df.to_parquet("user_data.parquet", index=False)
parquet_file = pq.ParquetFile("user_data.parquet")
print(f"Number of row groups: {parquet_file.metadata.num_row_groups}")
print(f"Number of columns: {parquet_file.metadata.num_columns}")
print(f"Number of rows: {parquet_file.metadata.num_rows}")
print(f"\nSchema:")
print(parquet_file.schema_arrow)

Number of row groups: 1
Number of columns: 8
Number of rows: 500000

Schema:
user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 992


In [6]:
first_group = parquet_file.metadata.row_group(0)

In [8]:
for i in range(first_group.num_columns):
    column_meta = first_group.column(i)
    stats = column_meta.statistics
    
    col_name = column_meta.path_in_schema
    phys_type = str(column_meta.physical_type)
    comp_size = column_meta.total_compressed_size
 
    if stats and stats.has_min_max:
        stat_str = f"Min: {stats.min}, Max: {stats.max}, Null: {stats.null_count}"
    else:
        stat_str = "couldn't find"

    print(f"{col_name:<15} | {phys_type:<15} | {comp_size:<22} | {stat_str}")

user_id         | INT64           | 2273849                | Min: 1, Max: 500000, Null: 0
city            | BYTE_ARRAY      | 252619                 | Min: Berlin, Max: Toronto, Null: 0
score           | DOUBLE          | 4272959                | Min: 5.188445665327279e-05, Max: 99.99983148609545, Null: 0
active          | BOOLEAN         | 63800                  | Min: False, Max: True, Null: 0
signup_date     | INT64           | 4272257                | Min: 2023-03-05 12:29:52.780118, Max: 2026-03-04 12:21:28.780118, Null: 0
age             | INT32           | 377947                 | Min: 18, Max: 80, Null: 0
sessions        | INT32           | 567226                 | Min: 0, Max: 500, Null: 0
revenue         | DOUBLE          | 4272959                | Min: 0.0009958058022618843, Max: 999.9978869129644, Null: 0


**Step 3: Save the same dataset as CSV. Compare file sizes. Print both sizes in KB and the compression ratio.**

In [13]:
import os

csv_file = "user_data.csv"
df.to_csv(csv_file, index=False)

parquet_size = os.path.getsize("user_data.parquet")
csv_size = os.path.getsize(csv_file)

parquet_kb = parquet_size / 1024
csv_kb = csv_size / 1024
compression_ratio = csv_size / parquet_size

print(f"CSV File Size: {csv_kb:.2f} KB")
print(f"Parquet File Size: {parquet_kb:.2f} KB")
print(f"Compression Ratio: {compression_ratio:.2f}x (Parquet is this much smaller than CSV)")

CSV File Size: 44484.07 KB
Parquet File Size: 15974.56 KB
Compression Ratio: 2.78x (Parquet is this much smaller than CSV)


**Step 4: Write a markdown cell explaining what information the Parquet metadata provides that CSV does not, and why that matters for query performance.**

* CSV files are just raw text; they have no idea what's inside them until the computer reads every single character. Parquet, however, stores a "table of contents" called metadata that tracks the exact data types of each column and the Minimum, Maximum, and Null Count for every block of data.
This is a game-changer for performance because of Row Group Skipping. If you search for users over age 70, the query engine looks at the metadata first—if a block says the "Max Age" is 40, it skips that entire section of the disk. Additionally, since the metadata points to exact byte locations for each column, the engine can jump straight to the data you need (like revenue) and ignore everything else. In a CSV, the computer is forced to scan every column in every row just to find the next comma. *

# Task 2

**Step 1: Read the full Parquet file from Task 1 and time it.
Step 2: Read only 2 columns from the same Parquet file and time it.**

In [15]:
df_parquet=pd.read_parquet('user_data.parquet').set_index('user_id')
%time df_parquet

CPU times: total: 0 ns
Wall time: 3.58 μs


,city,score,active,signup_date,age,sessions,revenue
user_id,,,,,,,
1,Sydney,43.689490,False,2025-01-13 10:30:35.780118,38,261,635.068666
2,London,97.403462,True,2024-01-28 02:48:56.780118,37,77,895.979998
3,Mumbai,66.148238,False,2023-12-07 19:19:43.780118,73,115,451.742873
4,Paris,21.993544,False,2026-01-08 23:22:39.780118,62,235,449.489098
5,Sydney,91.719953,True,2024-01-29 13:09:52.780118,20,399,100.266323
...,...,...,...,...,...,...,...
499996,London,61.531318,False,2025-03-27 18:34:41.780118,64,234,284.045041
499997,Berlin,74.712924,False,2026-02-25 07:39:27.780118,25,240,660.415732
499998,Toronto,48.430697,True,2023-06-06 12:18:48.780118,39,396,130.685450


In [17]:
%time df_parquet[['city','score']]

CPU times: total: 0 ns
Wall time: 5.19 ms


,city,score
user_id,,
1,Sydney,43.689490
2,London,97.403462
3,Mumbai,66.148238
4,Paris,21.993544
5,Sydney,91.719953
...,...,...
499996,London,61.531318
499997,Berlin,74.712924
499998,Toronto,48.430697


**Step 3: Read only 2 columns from the CSV file and time it (you will need to read the entire CSV and select columns after, which is what pandas does).**

In [22]:
csv_file=pd.read_csv('user_data.csv').set_index('user_id')
%time csv_file[['city','score']]

CPU times: total: 15.6 ms
Wall time: 4.43 ms


,city,score
user_id,,
1,Sydney,43.689490
2,London,97.403462
3,Mumbai,66.148238
4,Paris,21.993544
5,Sydney,91.719953
...,...,...
499996,London,61.531318
499997,Berlin,74.712924
499998,Toronto,48.430697


**Step 4: Write a markdown cell explaining why Parquet selective reads are faster. Connect your explanation to the column-chunk layout you observed in Task 1.**

* Parquet selective reads are faster because Parquet stores data in a columnar format, not a row-based format like CSV*

* Column Projection: Because the metadata stores the exact byte offsets for each column chunk, the system can "jump" directly to the columns requested in your query (e.g., only revenue) and completely ignore the rest of the file.
I/O Efficiency: Instead of loading the entire dataset into memory, the query engine only performs I/O operations on the specific chunks required. This reduces disk pressure and memory usage, especially when dealing with wide tables where you only need a fraction of the columns.*

# Task 3

**Step 1: Using PyArrow, read the Parquet file with a filter (e.g., age > 50). Time the read.**

In [26]:
import pyarrow.parquet as pq
%time pq.read_table("user_data.parquet",filters=[("age", ">", 50)])

CPU times: total: 31.2 ms
Wall time: 37.1 ms


pyarrow.Table
user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
----
user_id: [[3,4,6,7,8,...,131065,131066,131068,131069,131072],[131073,131074,131077,131080,131086,...,262140,262141,262142,262143,262144],[262145,262146,262150,262152,262153,...,393207,393210,393211,393213,393214],[393220,393221,393222,393225,393226,...,499994,499995,499996,499999,500000]]
city: [["Mumbai","Paris","Seoul","New York","Sydney",...,"Mumbai","Sydney","Tokyo","London","Tokyo"],["Toronto","San Francisco","Mumbai","Seoul","Berlin",...,"Mumbai","London","Berlin","Sydney","San Francisco"],["Seoul","Sydney","San Francisco","Mumbai","Mumbai",...,"Seoul","San Francisco","Sydney","New York","Seoul"],["Tokyo","New York","New York","Seoul","San Francisco",...,"Toronto","Toronto","London","Mumbai","Mumbai"]]
score: [[66.14823764798811,21.993544213770623,60.49715376221981,41.78996051330323,27.220311643727456,...,10.991213832005997,97.11478432394

**Step 2: Read the full Parquet file without a filter and apply the same filter in pandas after loading. Time this approach.**

In [27]:
%time pq.read_table("user_data.parquet")

CPU times: total: 31.2 ms
Wall time: 25 ms


pyarrow.Table
user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
----
user_id: [[1,2,3,4,5,...,131068,131069,131070,131071,131072],[131073,131074,131075,131076,131077,...,262140,262141,262142,262143,262144],[262145,262146,262147,262148,262149,...,393212,393213,393214,393215,393216],[393217,393218,393219,393220,393221,...,499996,499997,499998,499999,500000]]
city: [["Sydney","London","Mumbai","Paris","Sydney",...,"Tokyo","London","London","New York","Tokyo"],["Toronto","San Francisco","Toronto","New York","Mumbai",...,"Mumbai","London","Berlin","Sydney","San Francisco"],["Seoul","Sydney","London","San Francisco","London",...,"Tokyo","New York","Seoul","New York","London"],["New York","Paris","Paris","Tokyo","New York",...,"London","Berlin","Toronto","Mumbai","Mumbai"]]
score: [[43.689490033566045,97.40346189473134,66.14823764798811,21.993544213770623,91.71995305366835,...,73.70424820759605,77.26235287366444,78.811

In [33]:
table = pd.read_parquet("user_data.parquet")
%time table[table['age']>50]

CPU times: total: 0 ns
Wall time: 8.64 ms


,user_id,city,score,active,signup_date,age,sessions,revenue
2,3,Mumbai,66.148238,False,2023-12-07 19:19:43.780118,73,115,451.742873
3,4,Paris,21.993544,False,2026-01-08 23:22:39.780118,62,235,449.489098
5,6,Seoul,60.497154,False,2025-11-18 20:42:07.780118,74,367,552.469466
6,7,New York,41.789961,False,2025-10-05 01:39:22.780118,67,366,838.932934
7,8,Sydney,27.220312,True,2023-04-18 13:14:56.780118,55,421,357.700180
...,...,...,...,...,...,...,...,...
499993,499994,Toronto,91.227743,False,2024-10-21 07:03:47.780118,63,54,143.035877
499994,499995,Toronto,20.056590,True,2024-08-12 19:11:38.780118,54,45,341.728494
499995,499996,London,61.531318,False,2025-03-27 18:34:41.780118,64,234,284.045041
499998,499999,Mumbai,89.680757,True,2023-04-15 05:09:59.780118,60,273,441.912587


**Step 3: Compare the number of rows returned and the execution times. Write a markdown cell explaining what predicate pushdown does and why it is faster.**

* Comparison: Rows and Execution
Rows Returned: The query table['age'] > 50 returned 238,164 rows out of the original 500,000.
Execution Time: The "Wall time" was approximately 8.64 ms.  *

**Step 4: Run the same filtered query using DuckDB's SQL interface directly on the Parquet file:**

In [40]:
import duckdb
result = duckdb.sql("SELECT * FROM 'user_data.parquet' WHERE age > 50").df()
%time result

CPU times: total: 0 ns
Wall time: 4.53 μs


,user_id,city,score,active,signup_date,age,sessions,revenue
0,3,Mumbai,66.148238,False,2023-12-07 19:19:43.780118,73,115,451.742873
1,4,Paris,21.993544,False,2026-01-08 23:22:39.780118,62,235,449.489098
2,6,Seoul,60.497154,False,2025-11-18 20:42:07.780118,74,367,552.469466
3,7,New York,41.789961,False,2025-10-05 01:39:22.780118,67,366,838.932934
4,8,Sydney,27.220312,True,2023-04-18 13:14:56.780118,55,421,357.700180
...,...,...,...,...,...,...,...,...
238159,499994,Toronto,91.227743,False,2024-10-21 07:03:47.780118,63,54,143.035877
238160,499995,Toronto,20.056590,True,2024-08-12 19:11:38.780118,54,45,341.728494
238161,499996,London,61.531318,False,2025-03-27 18:34:41.780118,64,234,284.045041
238162,499999,Mumbai,89.680757,True,2023-04-15 05:09:59.780118,60,273,441.912587


**Task 4: pandas vs DuckDB — Identical Queries**

**Query 1: Count records per city.
Query 2: Average score by city, ordered by average score descending.
Query 3: For each city, find the percentage of active users whose score is above 75.
Query 4: Find the top 10 users by score for each city (window function / rank).
Query 5: Compute a running total of scores ordered by user_id, partitioned by city.**

In [51]:
result = duckdb.sql("SELECT count(city) as count_records,city FROM 'user_data.parquet' GROUP BY city").df()
%time result

CPU times: total: 0 ns
Wall time: 5.01 μs


,count_records,city
0,50061,Seoul
1,49982,Sydney
2,50059,Berlin
3,50059,San Francisco
4,49885,Mumbai
5,50216,Paris
6,50113,New York
7,49931,London
8,49724,Toronto
9,49970,Tokyo


In [49]:
table = pd.read_parquet("user_data.parquet")
%time table.groupby('city')['city'].count()

CPU times: total: 31.2 ms
Wall time: 32.1 ms


city
Berlin           50059
London           49931
Mumbai           49885
New York         50113
Paris            50216
San Francisco    50059
Seoul            50061
Sydney           49982
Tokyo            49970
Toronto          49724
Name: city, dtype: int64

In [66]:

result = duckdb.sql("""SELECT city, AVG(score) AS avg_score FROM 'user_data.parquet' GROUP BY city ORDER BY avg_score DESC""").df()
%time result

CPU times: total: 0 ns
Wall time: 4.77 μs


,city,avg_score
0,San Francisco,50.311448
1,Paris,50.151899
2,Tokyo,50.135854
3,Sydney,50.075511
4,Mumbai,50.063780
5,Berlin,50.013372
6,London,50.009568
7,Toronto,49.996347
8,Seoul,49.994487
9,New York,49.879502


In [68]:
%time  df.groupby('city')['score'].mean().sort_values(ascending=False).reset_index()


CPU times: total: 31.2 ms
Wall time: 47.8 ms


,city,score
0,San Francisco,50.311448
1,Paris,50.151899
2,Tokyo,50.135854
3,Sydney,50.075511
4,Mumbai,50.063780
5,Berlin,50.013372
6,London,50.009568
7,Toronto,49.996347
8,Seoul,49.994487
9,New York,49.879502


In [55]:

result = duckdb.sql("""SELECT 
    city, 
    100.0 * COUNT(CASE WHEN active = True AND score > 75 THEN 1 END) / COUNT(*) AS pct_high_active
FROM 'user_data.parquet' 
GROUP BY city;""").df()
result

,city,pct_high_active
0,Sydney,12.578528
1,Berlin,12.471284
2,San Francisco,12.695020
3,New York,12.334125
4,Tokyo,12.653592
5,Seoul,12.474781
6,London,12.677495
7,Toronto,12.468828
8,Mumbai,12.504761
9,Paris,12.452206


In [56]:
df['is_high_active'] = (df['active'] == True) & (df['score'] > 75)
result2 = (df.groupby('city')['is_high_active'].mean() * 100).reset_index()
print(result2)

            city  is_high_active
0         Berlin       12.471284
1         London       12.677495
2         Mumbai       12.504761
3       New York       12.334125
4          Paris       12.452206
5  San Francisco       12.695020
6          Seoul       12.474781
7         Sydney       12.578528
8          Tokyo       12.653592
9        Toronto       12.468828


In [61]:
result = duckdb.sql("""SELECT * FROM (
    SELECT 
        user_id, city, score, 
        RANK() OVER (PARTITION BY city ORDER BY score DESC) as rank
    FROM 'user_data.parquet'
)WHERE rank <= 10;""").df()
result

,user_id,city,score,rank
0,260671,Paris,99.999493,1
1,159526,Paris,99.996430,2
2,206509,Paris,99.994056,3
3,209114,Paris,99.993964,4
4,360624,Paris,99.993005,5
...,...,...,...,...
95,169685,San Francisco,99.986807,6
96,268858,San Francisco,99.986318,7
97,463773,San Francisco,99.982849,8
98,416775,San Francisco,99.981863,9


In [63]:
df['rank'] = df.groupby('city')['score'].rank(method='min', ascending=False)
result4 = df[df['rank'] <= 10]
result2

,city,is_high_active
0,Berlin,12.471284
1,London,12.677495
2,Mumbai,12.504761
3,New York,12.334125
4,Paris,12.452206
5,San Francisco,12.695020
6,Seoul,12.474781
7,Sydney,12.578528
8,Tokyo,12.653592
9,Toronto,12.468828


In [65]:
result = duckdb.sql("""SELECT 
    user_id, city, score, 
    SUM(score) OVER (PARTITION BY city ORDER BY user_id) AS running_city_score
FROM 'user_data.parquet';""").df()
result

,user_id,city,score,running_city_score
0,317966,Sydney,99.253834,1.593656e+06
1,317972,Sydney,29.426475,1.593686e+06
2,317988,Sydney,28.216848,1.593714e+06
3,318004,Sydney,80.467065,1.593794e+06
4,318036,Sydney,27.451420,1.593822e+06
...,...,...,...,...
499995,499958,New York,73.912225,2.499335e+06
499996,499968,New York,55.366415,2.499391e+06
499997,499974,New York,56.012926,2.499447e+06
499998,499976,New York,76.923640,2.499524e+06


In [64]:
df = df.sort_values(['city', 'user_id'])
df['running_city_score'] = df.groupby('city')['score'].cumsum()
print(df[['user_id', 'city', 'score', 'running_city_score']])

        user_id     city      score  running_city_score
21           22   Berlin  68.129047        6.812905e+01
25           26   Berlin  69.119490        1.372485e+02
51           52   Berlin  89.111342        2.263599e+02
78           79   Berlin  32.234163        2.585940e+02
83           84   Berlin  67.520144        3.261142e+02
...         ...      ...        ...                 ...
499991   499992  Toronto  14.317979        2.485774e+06
499992   499993  Toronto  84.212348        2.485859e+06
499993   499994  Toronto  91.227743        2.485950e+06
499994   499995  Toronto  20.056590        2.485970e+06
499997   499998  Toronto  48.430697        2.486018e+06

[500000 rows x 4 columns]


* 1. Ease of Expression
DuckDB (SQL): Much simpler for complex tasks like Ranking (Query 4) and Running Totals (Query 5). SQL’s window functions are more readable than Pandas' multi-step sorting and grouping logic.
Pandas: Very intuitive for simple aggregations like Averages (Query 2), but becomes "verbose" and harder to read as logic complexity increases.
2. Speed
DuckDB: Significantly faster. It processed a count/group operation in 3.34 µs, whereas Pandas took 32.1 ms—nearly 10,000 times slower for that specific task.
Reason: DuckDB is a vectorized engine; it processes data in chunks rather than row-by-row, which is far more efficient for large datasets.
3. Impact
Window Functions: This is where the difference matters most. In Query 5, the overhead of calculating cumulative sums for 500,000 rows across multiple cities is handled much more gracefully by DuckDB’s engine.
Scaling: While both handle 500,000 rows, DuckDB’s efficiency ensures it won't hit memory limits as quickly as Pandas when the data scales further. *

**Step 1: Create a pandas DataFrame and convert it to an Arrow Table using pyarrow.Table.from_pandas().
Step 2: Inspect the Arrow Table schema and compare it to the pandas dtypes. Note any differences.
Step 3: Write the Arrow Table to Parquet using pyarrow.parquet.write_table(). Read it back into a new Arrow Table.
Step 4: Convert the Arrow Table back to a pandas DataFrame using .to_pandas(). Verify the data matches the original.
Step 5: Demonstrate the pandas dtype_backend="pyarrow" option by reading the Parquet file with Arrow-backed dtypes. Print the dtypes and compare with traditional pandas dtypes.
Step 6: Write a markdown cell explaining the role of Arrow in the modern analytics stack. How does it connect Parquet (disk) to pandas (analysis) to DuckDB (SQL)?**

In [72]:
table = pa.Table.from_pandas(df)
table

pyarrow.Table
user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
is_high_active: bool
rank: double
running_city_score: double
__index_level_0__: int64
----
user_id: [[22,26,52,79,84,...,499992,499993,499994,499995,499998]]
city: [["Berlin","Berlin","Berlin","Berlin","Berlin",...,"Toronto","Toronto","Toronto","Toronto","Toronto"]]
score: [[68.12904743003625,69.11948952231505,89.11134203924568,32.234163003492654,67.52014363956332,...,14.317978579934898,84.2123478776331,91.22774331746936,20.056589787445734,48.430697138852295]]
active: [[true,true,true,true,true,...,false,true,false,true,true]]
signup_date: [[2023-06-15 09:49:39.780118000,2023-04-04 19:34:58.780118000,2023-05-10 00:04:03.780118000,2026-01-16 05:54:22.780118000,2026-02-14 06:06:18.780118000,...,2023-09-01 16:40:04.780118000,2024-07-06 21:14:36.780118000,2024-10-21 07:03:47.780118000,2024-08-12 19:11:38.780118000,2023-06-06 12:18:48.780118000]]
age: [[

In [73]:
print("Pandas Dtypes:\n", df.dtypes)
print("\nArrow Schema:\n", table.schema)

Pandas Dtypes:
 user_id                        int64
city                          object
score                        float64
active                          bool
signup_date           datetime64[ns]
age                            int32
sessions                       int32
revenue                      float64
is_high_active                  bool
rank                         float64
running_city_score           float64
dtype: object

Arrow Schema:
 user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
is_high_active: bool
rank: double
running_city_score: double
__index_level_0__: int64
-- schema metadata --
pandas: '{"index_columns": ["__index_level_0__"], "column_indexes": [{"na' + 1615


In [74]:
pq.write_table(table, 'arrow_data.parquet')
table_back = pq.read_table('arrow_data.parquet')

In [75]:
df_back = table_back.to_pandas()
print(f"\nIs information same? {df.equals(df_back)}")


Is information same? True


In [76]:
df_arrow_backed = pd.read_parquet('arrow_data.parquet', engine='pyarrow', dtype_backend='pyarrow')
print("\nTraditional Pandas Dtypes (Sessions):", df['sessions'].dtype)
print("Arrow-backed Pandas Dtypes (Sessions):", df_arrow_backed['sessions'].dtype)


Traditional Pandas Dtypes (Sessions): int32
Arrow-backed Pandas Dtypes (Sessions): int32[pyarrow]


* The Role of Apache Arrow in the Modern Analytics Stack
Apache Arrow acts as the universal language (lingua franca) for data in memory, enabling seamless communication between different tools without the slow process of "translation" (serialization).
Parquet to RAM: Parquet stores data column-by-row on the disk. Arrow reads this directly into memory using a matching columnar format, allowing for zero-copy reads that save time and CPU.
Arrow to Pandas: Historically, Pandas used NumPy, but with the pyarrow backend, it can now process Arrow tables directly. This eliminates the need to convert data types, reducing memory overhead.
Arrow to DuckDB: DuckDB is a vectorized engine that "speaks" Arrow natively. It can run SQL queries directly on Arrow tables in microseconds (µs) because the data is already in the optimal format for analytical processing.*